# 06 - Depth-Only Ablation
Train LSTM on **depth-only** features to measure depth modality's standalone contribution.

**Purpose:** Compare depth-only vs IR-only (Run 7b k=2: MPCA 52.47%) and fusion (Run 8: MPCA 56.51%) to quantify each modality's individual contribution.

**Pre-requisite:** Depth merged features must exist in `features_depth_merged_k2/` (from Run 8 pipeline).

**Runtime:** ~5 min on Colab T4 (feature-based, no CNN/extraction needed).

In [ ]:
# Colab Setup
import os
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/Driver-Activity-Recognition'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/batuhne/Driver-Activity-Recognition.git {REPO_DIR}

    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
    DATA_ROOT = '/content/drive/MyDrive/DriveAndAct'
else:
    DATA_ROOT = './data'

print(f'Working directory: {os.getcwd()}')
print(f'Data root: {DATA_ROOT}')

In [ ]:
# Verify depth features exist
import csv
import numpy as np

depth_feat_dir = os.path.join(DATA_ROOT, 'features_depth_merged_k2')

for split in ['train', 'val', 'test']:
    manifest_path = os.path.join(depth_feat_dir, split, 'manifest.csv')
    with open(manifest_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    sample_path = os.path.join(depth_feat_dir, split, rows[0]['filename'])
    sample = np.load(sample_path)

    print(f"{split}: {len(rows)} segments, shape={sample.shape}")
    assert sample.shape == (32, 512), f"Shape mismatch: {sample.shape}"

print('\nDepth features verified OK')

In [ ]:
# Backup Run 8 best_model before training overwrites it
checkpoint_dir = os.path.join(DATA_ROOT, 'results', 'checkpoints')

run8_model = os.path.join(checkpoint_dir, 'best_model.pth')
run8_backup = os.path.join(checkpoint_dir, 'best_model_run8.pth')

if os.path.exists(run8_model) and not os.path.exists(run8_backup):
    import shutil
    shutil.copy2(run8_model, run8_backup)
    print(f'Run 8 model backed up: {run8_backup}')
elif os.path.exists(run8_backup):
    print(f'Backup already exists: {run8_backup}')
else:
    print('WARNING: best_model.pth not found')

In [ ]:
# Train depth-only LSTM
import torch
import shutil
from src.utils import load_config
from src.train import train

config = load_config()
if IN_COLAB:
    config['data']['root'] = DATA_ROOT
    drive_output = os.path.join(DATA_ROOT, 'results')
    config['output']['checkpoint_dir'] = os.path.join(drive_output, 'checkpoints')
    config['output']['log_dir'] = os.path.join(drive_output, 'logs')
    config['output']['figure_dir'] = os.path.join(drive_output, 'figures')

# === DEPTH-ONLY CONFIG ===
config['features']['save_dir'] = 'features_depth_merged_k2'  # depth features only
# No fusion_dir — single stream
config['model']['feature_dim'] = 512                          # depth only, not 1024
config['training']['mode'] = 'feature_based'

# Model config — identical to Run 7b k=2 and Run 8
config['model']['use_layernorm'] = True
config['model']['bidirectional'] = True
config['model']['pooling'] = 'attention'
config['model']['lstm_hidden'] = 256
config['model']['lstm_dropout'] = 0.3

# Training config — identical to previous runs
config['training']['batch_size'] = 32
config['training']['lr'] = 0.001
config['training']['loss_type'] = 'ce'
config['training']['label_smoothing'] = 0.1
config['training']['mixup_alpha'] = 0.0
config['training']['noise_std'] = 0.0
config['training']['weight_decay'] = 0.0001
config['training']['epochs'] = 50
config['training']['early_stop_patience'] = 12
config['training']['scheduler_type'] = 'plateau'
config['training']['scheduler_factor'] = 0.5
config['training']['scheduler_patience'] = 5
config['training']['gradient_clip'] = 1.0
config['training']['use_weighted_sampler'] = True
config['training']['en_beta'] = 0.99
config['training']['num_workers'] = 2

# --- Delete old resume files to ensure fresh run ---
checkpoint_dir = config['output']['checkpoint_dir']
log_dir = config['output']['log_dir']

for stale_file in ['last_checkpoint.pth', 'best_model.pth']:
    path = os.path.join(checkpoint_dir, stale_file)
    if os.path.exists(path):
        os.remove(path)
        print(f'Deleted: {path}')

if os.path.exists(log_dir):
    shutil.rmtree(log_dir)
    print(f'Deleted log dir: {log_dir}')

print(f'\n--- Depth-Only Ablation Config ---')
print(f"Features: {config['features']['save_dir']}")
print(f"Feature dim: {config['model']['feature_dim']}")
print(f"LSTM: h={config['model']['lstm_hidden']}, BiLSTM={config['model']['bidirectional']}, pool={config['model']['pooling']}")
print(f"LR: {config['training']['lr']}, batch={config['training']['batch_size']}")
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Train the model
model = train(config)

In [ ]:
# GPU memory summary
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1024**3
    current_mem = torch.cuda.memory_allocated() / 1024**3
    total_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f'GPU Memory: peak={peak_mem:.2f} GB, current={current_mem:.2f} GB, total={total_mem:.1f} GB')
    print(f'Utilization: {peak_mem/total_mem*100:.0f}%')

## Monitor Training

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {config['output']['log_dir']}